# 🚀 Kaggle Max — Đại Mô Hình Suy Luận **32B AWQ** (`Qwen3-32B-AWQ` / `QwQ-32B-AWQ`) Trên **Dual Tesla T4 (`2 × 16 GB = 32 GB VRAM`)**
Notebook này triển khai mô hình **32B 4-bit AWQ** (~18.4 GB trọng số, chia đều ~9.2 GB/GPU bằng **Tensor Parallelism `TP=2`**) trên 2 card NVIDIA Tesla T4 (`sm_75`, `dtype=float16`).
- **Mặc định khuyên dùng:** `Qwen/Qwen3-32B-AWQ` (Hỗ trợ linh hoạt `/no_think` cho bước gọi Tool nhanh và `/think` cho bước Tổng hợp & Kiểm chứng sâu).
- **Tùy chọn thay thế:** Đổi `KAGGLE_MAX_MODEL = 'Qwen/QwQ-32B-AWQ'` nếu muốn chạy CoT toàn phần.

In [ ]:
!pip install -q vllm fastapi uvicorn pyngrok httpx

In [ ]:
import os, subprocess, time, torch
print('GPUs detected:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2), 'GB')

# Chọn 'Qwen/Qwen3-32B-AWQ' (khuyên dùng số 1) hoặc 'Qwen/QwQ-32B-AWQ'
KAGGLE_MAX_MODEL = os.environ.get('KAGGLE_MAX_MODEL', 'Qwen/Qwen3-32B-AWQ')
NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')  # Điền Ngrok Authtoken của bạn hoặc dùng Cloudflare Tunnel bên dưới

In [ ]:
# Khởi chạy vLLM Server với Tensor Parallelism = 2 trên 2x Tesla T4 (cổng 8000 nội bộ)
vllm_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', KAGGLE_MAX_MODEL,
    '--tensor-parallel-size', '2',
    '--dtype', 'float16',
    '--quantization', 'awq',
    '--enforce-eager',
    '--gpu-memory-utilization', '0.90',
    '--max-model-len', '8192',
    '--port', '8000',
]
vllm_proc = subprocess.Popen(vllm_cmd)
print('⏳ Đang khởi động vLLM 32B TP=2 trên cổng 8000...')

In [ ]:
# FastAPI Gateway (cổng 8080) tương thích đồng thời cả /generate, /health và /v1/chat/completions
gateway_code = '''
import os, httpx, uvicorn
from fastapi import FastAPI, Request

app = FastAPI(title='Kaggle Max 32B TP=2 Gateway')
VLLM_URL = 'http://127.0.0.1:8000'
MODEL_ID = os.environ.get('KAGGLE_MAX_MODEL', 'Qwen/Qwen3-32B-AWQ')
client = httpx.AsyncClient(timeout=180.0)

@app.get('/health')
async def health():
    return {'status': 'online', 'model': MODEL_ID, 'has_vllm_32b': True, 'tp_size': 2}

@app.post('/generate')
@app.post('/api/generate')
async def generate(req: Request):
    body = await req.json()
    prompt = body.get('prompt', '')
    sys_inst = body.get('system_instruction') or 'Bạn là Đại Mô Hình Nghiên Cứu Khoa Học 32B.'
    messages = [{'role': 'system', 'content': sys_inst}, {'role': 'user', 'content': prompt}]
    payload = {
        'model': MODEL_ID,
        'messages': messages,
        'temperature': body.get('temperature', 0.2),
        'max_tokens': body.get('max_tokens', 2048),
    }
    resp = await client.post(f'{VLLM_URL}/v1/chat/completions', json=payload)
    data = resp.json()
    text = data.get('choices', [{}])[0].get('message', {}).get('content', '')
    return {'text': text, 'output': text, 'model': MODEL_ID}

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8080)
'''
with open('gateway_32b.py', 'w', encoding='utf-8') as f:
    f.write(gateway_code)
gw_proc = subprocess.Popen(['python', 'gateway_32b.py'])
print('✅ Đã bật Gateway 32B tại cổng 8080!')